# 08a · PyTorch 入门：ESM 向量 + 一个小网络

## 这一章在整个项目里的位置

```
00–07 ✅ → [08 深度学习] → 09 全蛋白组预测 → 10 对外 API
              │
              ├─ 08a PyTorch 入门：ESM 向量 + 小网络   ← 你在这里
              ├─ 08b 图神经网络：AlphaFold 结构图
              └─ 08c 微调 ESM-2（LoRA）
```

## 背景：为什么需要这一章

07f 的基线留下了一个明确的结论：**稳定性学会了，功能没学会**。它用的是 16 个我们亲手挑的特征，其中来自 ESM-2 的只有 3 个数：突变打分、位置的熵、原氨基酸的概率。

但 ESM-2 对每个残基其实给了 **1,280 个数**（07e 存在 `silver.residue_embedding` 里的向量）。那 3 个数是我们从里面"摘要"出来的，剩下的信息全扔了。这一章把整个向量交给模型，让它自己决定哪些有用。

1,280 个数单独看哪一个都没有意义，合起来才有意义。处理这种输入是**神经网络**的本行。写神经网络的标准工具是 **PyTorch**，这一章从零用它搭第一个网络。

这一章也是 08 整个部分的地基：08b 的图神经网络、08c 的微调，用的都是这里学的同一套训练办法。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 训练全在你的 Mac 上，用 Mac 自带的 GPU。一个网络完整考一遍（5 折）不到一分钟
   └─ 成绩、参数、模型记到 Databricks 的 MLflow，和 07f 的基线记在同一个本子里
看结果：这里的输出；Databricks 网页的 Experiments
```

**为什么不在 Databricks 上训练**：07e 遇到过，免费版的 GPU 配额是 0。而这个网络很小，Mac 的 GPU 绰绰有余。**在哪里训练不要紧，要紧的是记录在同一个地方、考法是同一套**。公司里也常这样：开发机上训练，中央的 MLflow 记账。

## 这一章做的 7 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | PyTorch 是什么 | 会自己求导的计算器 |
| 2 | 把训练数据取到本机 | 从库房领料到实验台 |
| 3 | 网络长什么样 | 在 ESM-2 这台大仪器后面接一个小读数头 |
| 4 | 训练一次，看着它学 | 背题和学会的区别 |
| 5 | 正式考试，记进 MLflow | 同一张卷子，和基线比 |
| 6 | 看结果 | 稳定性又涨了，功能呢？ |
| 7 | 合起来用 | 三个大夫会诊 |

**花费：0**。**时间**：约 10 分钟，其中训练加起来约 5 分钟。

---

## 准备

运行下面这一格：连上 Databricks，载入这一章的代码，看看 PyTorch 找到了什么处理器。**不用看代码。**

In [ ]:
import os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")
import pandas as pd
import torch
from databricks.sdk import WorkspaceClient
from qafi_platform import lakehouse, tracking
from qafi_platform.lakehouse import sql
from qafi_core import baseline, neural
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"   # 和 07f 是同一个 MLflow 实验
ESM2 = "08e4846e537177426273712802403f7ba8261b6c"                            # 07e 钉死的 ESM-2 版本
MODELS = ROOT / "data/models"                                                # 本机的缓存，不进 Git
MODELS.mkdir(parents=True, exist_ok=True)
print("Databricks:", w.config.host)
print("PyTorch", torch.__version__, "| 训练用的处理器:", neural.device())

### 结果怎么看
最后一行应该是 `mps`：这是 PyTorch 对 Apple 芯片 GPU 的叫法（Metal Performance Shaders）。07e 读 ESM-2 时用的就是它。如果是 `cpu`，也能跑，只是慢几倍。

---

## 第 1 步 · PyTorch 是什么

### 在干什么
弄清楚 PyTorch 提供了哪几样东西，训练一个网络到底在做什么。

### 背景
比喻：**校准仪器**。一台新仪器有很多旋钮。你放一个已知浓度的标准品进去，看读数差多少，把旋钮拧一点；再放一个，再拧。几千次以后，读数就准了。

训练神经网络是同一件事，只是规模大得多：
- **旋钮** = 网络的**参数**（parameters）。我们的小网络有 73 万个
- **标准品** = 训练数据：已知答案的突变
- **读数差多少** = **损失**（loss）：预测和答案差距的一个数，越小越好
- **每个旋钮该往哪边拧** = **梯度**（gradient）：损失对每个参数的导数
- **拧一点** = 沿着梯度的反方向，把每个参数改一小步

**先看只有 1 个旋钮的时候**。假设模型只有一个数 `w`，规则是：预测 = `w` × 输入。有一道题：输入是 2，正确答案是 10。`w` 一开始随便给个 1。

1. **猜**：1 × 2 = 2
2. **对答案**：答案是 10，差了 8。差得越多，损失越大
3. **找方向**：把 `w` 调大一点点试试，改成 1.1，预测变成 2.2，只差 7.8 了。变好了，所以 `w` 该**往大了调**。"该往哪边调、调了管多大用"，这个信息就是**梯度**
4. **改一点**：把 `w` 往大调一小步，比如调到 1.5

回到第 1 步重来。重复几十次，`w` 会走到 5（5 × 2 = 10），就调对了。**训练神经网络从头到尾就是这四个动作在重复：猜 → 对答案 → 找方向 → 改一点。**

**73 万个旋钮的时候**，难在第 3 步。1 个旋钮可以"调大一点试试看"；73 万个旋钮每个都试一遍，要算 73 万次，而这只是走一步。**PyTorch 最核心的本事就是自动求导**：算一次，就同时告诉你 73 万个旋钮各自该往哪边调。你只管写"怎么从输入算到损失"，剩下的它来。

你更熟悉的类比是调显微镜的焦距：拧一下，更清楚就继续往这边拧。"糊的程度"是损失，"往哪边拧会更清楚"是梯度。训练网络是同时调 73 万个调焦旋钮。

PyTorch 给的四样东西，这一章全用上：

| 名字 | 大白话 | 对应上面哪一步 | 这一章里的例子 |
|---|---|---|---|
| **Tensor**（张量） | 装数字的表格，像 Excel 的一张表，但可以放到显卡上算 | 题目和答案都装在里面 | 29 万 × 1,280 的向量表 |
| **autograd**（自动求导） | `loss.backward()` 一句话，找出所有旋钮的方向 | 第 3 步 | 训练循环里每一步都调用 |
| **nn.Module** | 模型本身：那一堆旋钮，加上它们怎么连在一起 | 第 1 步用它来猜 | 我们的 `Head` |
| **optimizer**（优化器） | 真正动手去拧旋钮的那个人 | 第 4 步 | `AdamW` |

**这是监督学习**：每道题都有标准答案（07c 的标签），网络猜完可以对答案。ESM-2 自己当初不是这样训练的：它遮住序列里的氨基酸让自己猜，答案就是原序列，没有人给它标注，那叫**自监督学习**。

### 执行
先亲眼看一次自动求导。让 y = x² + 2x，在 x = 3 这一点，导数应该是 2x + 2 = 8：

In [ ]:
x = torch.tensor(3.0, requires_grad=True)   # 一个要被求导的数
y = x ** 2 + 2 * x                          # 随便算点什么，PyTorch 在背后记下了算的过程
y.backward()                                # 倒着走一遍这个过程，求出导数
print("y =", y.item(), "| dy/dx =", x.grad.item())

### 结果怎么看
`dy/dx = 8.0`，和手算的一样。把这里的 x 换成 73 万个参数、把 y 换成损失，就是训练神经网络时每一步在做的事。`backward` 这个名字的意思是"倒着传回去"，所以这个算法叫**反向传播**（backpropagation）。

### 用了什么技术，为什么
**PyTorch**（Meta 开源）是今天深度学习研究和工业界的事实标准：ESM-2、AlphaFold 的开源复现、Hugging Face 上绝大多数模型都是用它写的。另一个老牌框架 TensorFlow 在新项目里已经很少见。学会 PyTorch，别人的模型代码你就都读得懂。

### 面试怎么说
> "Training is a loop of four steps: forward pass, loss, backward pass to get gradients by automatic differentiation, and an optimizer step. PyTorch gives tensors that live on the GPU, autograd, composable modules and optimizers."

---

## 第 2 步 · 把训练数据取到本机

### 在干什么
从平台的表里取两样东西到你的 Mac：训练用的突变，和它们所在残基的 ESM 向量。

### 背景
比喻：**从库房领料**。数据的正本在平台的表里（有版本、有测试、有血缘），训练时领一份到实验台上用。领的是哪一版，要记下来。

两样东西：

| 取什么 | 从哪张表 | 多大 |
|---|---|---|
| 训练突变：特征、标签、分在哪一折 | `gold.fct_variant_features`，只取 `split = 'train'` | 84 万行，20 MB |
| 残基的向量 | `silver.residue_embedding` | 29 万个残基 × 1,280 个数 |

**考试蛋白一行都不取。** SQL 里写死了 `split = 'train'`，考试蛋白的标签根本到不了这台电脑，想偷看也看不到。

### 用了什么技术，为什么
- **`lakehouse.download`**：以前用的 `sql()` 把结果当文字传回来，上限 25 MB，取不了这么多。`download` 让 SQL 仓库把结果打包成 **Arrow** 文件（一种列式的二进制格式，数字不用转成文字），直接存成 parquet。大结果的标准取法
- **向量拼成一个大矩阵**：07e 存的是一个残基一行、每行一个长度 1,280 的列表，适合存，不适合训练。`neural.embedding_table` 把它们拼成一个 29 万 × 1,280 的矩阵存在磁盘上，另外记一张"哪个蛋白的哪个位置在第几行"的索引。只做一次，以后直接打开
- **16 位浮点数**（float16）：每个数占 2 个字节而不是 4 个，矩阵从 1.5 GB 变成 0.75 GB。向量的数值大多在 ±1 以内，16 位能保留三位有效数字，足够了

### 执行
先取训练突变：

In [ ]:
lakehouse.download(w, f"SELECT * FROM {CATALOG}.gold.fct_variant_features WHERE split = 'train'", MODELS / "train_variants.parquet")
train = pd.read_parquet(MODELS / "train_variants.parquet")
VERSION = sql(w, f"DESCRIBE HISTORY {CATALOG}.gold.fct_variant_features LIMIT 1")["version"][0]
print(f"{len(train):,} 个训练突变，{train.uniprot_id.nunique()} 个蛋白，{train.dataset_id.nunique()} 组实验；特征表版本 {VERSION}")

再准备向量。07e 第 5 步在你 Mac 上算出来的那批文件还在的话，直接用；换了电脑没有这些文件，就从平台的表里取。两条路我都走过，得到的矩阵一模一样：

In [ ]:
parts = ROOT / f"data/esm2-650m/{ESM2}-labelled/embeddings"   # 07e 在这台 Mac 上算出来的文件
if not parts.exists():                                        # 没有：从平台的表里取（1.5 GB，约 5 分钟）
    parts = MODELS / "embeddings.parquet"
    lakehouse.download(w, f"""SELECT e.uniprot_id, e.position, e.embedding FROM {CATALOG}.silver.residue_embedding e
                              JOIN {CATALOG}.gold.protein_split s USING (uniprot_id) WHERE s.split = 'train'""", parts)
table, index = neural.embedding_table(parts, train.uniprot_id.unique(), MODELS / "train_embeddings.npy")
print("向量表:", table.shape, table.dtype, f"| {table.nbytes / 1e9:.2f} GB")
index.head(3)

### 结果怎么看
- **841,400** 个训练突变、**349** 个蛋白、**445** 组实验，和 07f 第 3 步的数字一致
- 向量表 **293,265 行 × 1,280 列**，`float16`，0.75 GB：训练蛋白的每个残基一行
- 下面的小表是索引：`A0PJY2` 这个蛋白的第 1 个残基在矩阵的第 0 行，第 2 个在第 1 行……

注意 84 万个突变只对应 29 万个残基：一个位置能换成好几种氨基酸，它们**共用同一个向量**。向量描述的是"这个位置"，不是"这次替换"。

### 公司里什么情况怎么做
- 训练数据从**治理过的表**里取，并且记下版本号。从谁的硬盘上拷一份 CSV 来训练，是结果无法复现的头号原因
- 数据大到内存放不下时，才需要 PyTorch 的 `Dataset` 和 `DataLoader`：它们一批一批地从磁盘读。我们这里 0.75 GB，整个放进 GPU 更快，就不绕这个弯了。08c 微调 ESM-2 时序列长短不一，会用上 `DataLoader`

### 面试怎么说
> "Training data is pulled from versioned tables with the split filter in the query, so test labels never reach the training machine. The per-residue embeddings are packed once into a memory-mapped float16 matrix with a residue index, and variants look their residue up by row."

---

## 第 3 步 · 网络长什么样

### 在干什么
看清楚这个网络的每一层在干什么。

### 背景
比喻：**在大仪器后面接一个小读数头**。ESM-2 是那台大仪器，6.5 亿个参数，别人花了几百万美元的电费训练好的。我们**一个参数都不动它**（这叫**冻结**，frozen），只在它的输出后面接一个小网络，让小网络学会"怎么读这 1,280 个数"。这种做法叫**迁移学习**（transfer learning）：用别人训练好的大模型的理解力，自己只训练最后一小段。

```
一个残基的 ESM 向量（1,280 个数）
        │  LayerNorm：把这 1,280 个数拉到同一个量级
        ▼
   [ 拼上 07f 的 16 个特征 ]            ← 可选：三次实验里有两次拼了
        │  Linear 1,296 → 512，GELU，Dropout
        │  Linear 512 → 128，GELU，Dropout
        ▼
   Linear 128 → 20                     ← 20 个输出：这个位置换成 20 种氨基酸，各打多少分
        │
        ▼
   取"换成的那一种"的那个分数           ← 比如 R175H，就取 H 那一个
```

**拿一个突变从头走一遍。** 某个蛋白第 50 位是 R，突变成 H，问功能还剩多少：

1. **输入**：ESM-2 给第 50 位写的 1,280 个数。它描述的是"这个位置"，和换成什么无关
2. **LayerNorm，统一量级**：这串数里要是有一个特别大，后面全被它带着跑。先整体缩放一下，相当于做一次 z-score
3. **拼上 16 个特征**：把 `rsa`、`plddt` 这些接在后面，1,280 + 16 = 1,296 个数
4. **第一层，512 个"评委"初审**：每个评委都看全部 1,296 个数，给每个数一个权重，加起来得到自己的一个分。每个评委的权重不同，关注的东西就不同：训练完以后，可能有的在看"是不是埋在疏水核心"，有的在看"像不像结合位点"。没人教它们看什么，是训练时自己调出来的
5. **GELU，评委要么发言要么沉默**：算出的分是负的，基本压成 0；是正的，留下
6. **Dropout，训练时随机让 20% 的评委请假**：剩下的人必须把活顶上，网络就不会只靠某一两个评委。正式预测时全员到场
7. **第二层，128 个评委复审**：他们看的不再是原始数字，而是初审 512 个评委的意见
8. **最后一层，20 个终审**：一人管一种氨基酸。输出 20 个分数：换成 A 几分、换成 C 几分……换成 H 是 −1.3
9. **取 H 那一个**：−1.3 就是对 R→H 的预测。训练时拿它和标签比，差多少，回头去调前面所有评委的权重

这和一条信号通路很像：很多上游信号进来，第一层激酶整合，过了阈值才往下传，再传给第二层，最后激活不同的下游输出。Dropout 相当于随机敲掉几个激酶，逼着通路有冗余。

每种积木一句话：

| 积木 | 干什么 |
|---|---|
| **Linear**（线性层） | 把输入的每个数乘上一个权重再加起来，得到新的一组数。权重就是要训练的"旋钮" |
| **GELU**（激活函数） | 在两个线性层之间加一道"弯"。没有它，叠多少层线性层都等于一层，学不了复杂的关系 |
| **Dropout** | 训练时每次随机关掉 20% 的单元。逼着网络不依赖任何一个单元，减轻死记硬背 |
| **LayerNorm** | 把一个残基的 1,280 个数整体缩放到均值 0、方差 1，训练更稳 |

**为什么输出是 20 个数，而不是 1 个**：一个位置有 19 种可能的替换。让网络看一次这个位置，同时回答"换成 A 怎样、换成 C 怎样……"，比每种替换各算一遍省事，而且逼着它先理解这个位置本身。ESM-2 自己的输出也是这个形状（每个位置 20 个概率）。

### 执行
把网络打印出来，数一数参数：

In [ ]:
head = neural.Head(mean=torch.zeros(16), std=torch.ones(16))   # 拼 16 个特征的那一版
print(head)
print(f"\n{sum(p.numel() for p in head.parameters()):,} 个要训练的参数")

### 结果怎么看
- 打印出来的就是上面那张图：`norm`（LayerNorm）、`body`（两层 Linear，中间夹着 GELU 和 Dropout）、`out`（最后那层，输出 20 个数）
- **734,868 个参数**。ESM-2 有 6.5 亿个，我们训练的这一小段只有它的千分之一
- 大头在第一层：(1,280 + 16) × 512 ≈ 66 万个

代码在 [`src/qafi_core/neural.py`](../../src/qafi_core/neural.py) 的 `Head` 这个类里，一共 20 行。写一个 PyTorch 模型就是两件事：`__init__` 里声明有哪些层，`forward` 里写数据怎么流过它们。

### 用了什么技术，为什么
- **冻结大模型，只训练一个头**：便宜（不用 GPU 集群）、数据要得少（73 万个参数，不是 6.5 亿个）、不容易学坏。这是用预训练模型的第一选择。只有当它不够好时，才去动大模型本身，那是 08c 的事
- **这算微调吗？不算。** 判断标准只有一条：大模型自己的参数变了没有。这里 ESM-2 一个参数都没动，向量在 07e 算好就固定了，这叫**特征提取**。**微调**是让标签的信号一路传回 ESM-2 内部，把它自己的参数也改掉，同一个残基算出来的向量会变。两种都属于迁移学习。打个比方：ESM-2 是一位写好了笔记的专家；特征提取是另找一个新人学着读他的笔记，专家本人什么也没学到；微调是把题目和答案拿给专家本人，让他自己改进
- **同样做法的已发表模型**：ThermoMPNN（冻结 ProteinMPNN，后接小网络，预测稳定性）、VESPA 和 VespaG（语言模型的向量，后接小模型）、VariPred（ESM-1b 的向量，后接分类器，预测致病性）。所以这个结构本身是标准做法，不是我们的发明
- **把特征的缩放存在模型里**：16 个特征的量纲差得远（pLDDT 是 0–100，rsa 是 0–1），进网络前要各自减均值、除标准差。这两组数用**训练数据**算出来，然后**存进模型**。这样模型拿到新数据时，缩放方式和训练时一模一样。这又是 07a 说的 training-serving skew

### 面试怎么说
> "The model is a frozen ESM-2 with a small trainable head: layer norm, two hidden layers with GELU and dropout, and 20 outputs per residue, one per possible amino acid, of which the variant's is picked. Feature scaling constants are stored as buffers in the model so inference scales inputs exactly as training did."

---

## 第 4 步 · 训练一次，看着它学

### 在干什么
训练网络 10 轮，每一轮都看两个数：它对训练题答得多好，对没见过的蛋白答得多好。

### 背景
比喻：**背题和学会的区别**。一个学生把练习册做了十遍，练习册上的题全对。这说明他学会了吗？不一定，可能只是把答案背下来了。要知道他是不是真会，得拿**他没做过的题**来考。

网络也一样。训练循环长这样（`neural.train`，核心就 6 行）：

```
重复 N 轮（epoch）：                      一轮 = 把所有训练突变过一遍
    把训练突变打乱，切成 4,096 个一批（batch）
    对每一批：
        预测 = 网络(这一批)              ← 前向（forward）
        损失 = 预测和答案的加权平方差     ← 和 07f 一样，每组实验一票
        损失.backward()                  ← 反向传播：算出 73 万个梯度
        优化器.step()                    ← 每个参数沿梯度反方向改一小步
```

几个要定的数（叫**超参数**，hyperparameter：不是网络自己学的，是人定的）：

| 名字 | 我们用的 | 意思 |
|---|---|---|
| **batch size** | 4,096 | 每看多少道题改一次参数 |
| **learning rate**（学习率） | 0.0003 | 每次改多大一步。太大学不稳，太小学得慢 |
| **epochs**（轮数） | 4 | 把训练数据看几遍。**这一步就是要看这个数怎么定** |
| **AdamW** | | 现在最常用的优化器：自动给每个参数调步子大小 |

### 执行
拿第 0 折的蛋白当"没做过的题"，在其余四折上训练 10 轮（约半分钟）：

In [ ]:
curve = neural.learning_curve(train, table, index, features=neural.FEATURES, standardise=True, settings={"epochs": 10})
curve.plot(subplots=True, marker="o", figsize=(7, 4.5), xlabel="epoch",
           title=["training loss (lower = fits the training proteins better)", "score on held-out proteins (higher = better)"])
curve.T

### 结果怎么看
- **上图，训练损失**：一路往下，10 轮都没停。网络对训练题越答越好
- **下图，没见过的蛋白上的成绩**：升到**第 4 轮**前后见顶（约 0.63），然后**一路往下掉**，第 10 轮只剩 0.59 左右

两条线从第 4 轮开始分道扬镳。**训练题越答越好，新题却越答越差**，这就是**过拟合**（overfitting）：网络开始记住训练蛋白各自的脾气，而不是学通用的规律。

所以轮数定在 **4**。这个数是这样选出来的，不是拍脑袋。

### 公司里什么情况怎么做
- **永远画这两条线**。只看训练损失，会以为训练得越久越好。很多"模型上线后效果差"的事故，根子就是只盯着训练损失
- 在最好的那一轮停下来，叫**早停**（early stopping）。严格的做法是从训练数据里再分一小块专门用来决定何时停；我们这里用交叉验证的一折来看，考试蛋白始终没碰
- 学习率、层的宽度、dropout 我都试过几组：学习率 0.001 和 0.0003、dropout 0.2 和 0.4、换三个随机种子，整体成绩都在 0.627 上下 0.005 以内。**这个网络对这些设置不敏感，对训练轮数敏感**

### 面试怎么说
> "I track training loss and held-out score per epoch. The loss kept falling for ten epochs while the score on held-out proteins peaked at epoch four and then dropped: overfitting. So training stops at four epochs; other hyperparameters barely mattered."

---

## 第 5 步 · 正式考试，记进 MLflow

### 在干什么
和 07f 一模一样的考法考三个网络，每个都记进 MLflow。

### 背景
三个网络，后一个比前一个多一样东西，这样每样东西值多少一目了然（这就是 07f 说的消融实验，倒过来做）：

| 名字 | 输入 | 标签 |
|---|---|---|
| `mlp_embedding` | 只有 ESM 向量 | 原样 |
| `mlp_embedding_features` | 向量 + 07f 的 16 个特征 | 原样 |
| `mlp_standardised` | 向量 + 16 个特征 | **每组实验各自标准化** |

（MLP 是 multi-layer perceptron 的缩写，就是这种"几层线性层叠起来"的网络最老的名字。）

**"每组实验各自标准化"是什么**：07c 把所有实验换算到了同一把尺子上（1 = 野生型，0 = 失活），但每组实验的分数**散得多开**还是不一样，有的挤在 0.8–1.0，有的铺满 0–1。网络为了压低平方误差，会花力气去猜"这组实验的刻度有多宽"，而这件事对新蛋白毫无用处。把每组实验的标签各自减均值、除标准差，网络就只需要学**同一组实验里谁高谁低**，这正是我们考的东西（Spearman 只看排序）。

### 用了什么技术，为什么
**同一张卷子**：
- 折是同一个函数分的（`validation.group_folds`），所以和 07f 的基线**每一折的蛋白完全相同**
- 成绩是同一个函数算的（`baseline.dataset_scores`）
- 所以两个模型在每组实验上的分数可以**一对一配对**比较

**在本机训练，在平台记账**（`src/qafi_platform/tracking.py`）。每个运行留下和 07f 一样的记录：
- MLflow 里一页：参数、成绩、和两根横杆（零样本、基线）的配对比较
- `ml.cv_scores` 里每组实验一行
- 标签里多两项：`trained_on = laptop`，以及代码版本。**如果你有没提交的改动，版本号后面会带 `-dirty`**，以后一看就知道这次结果对应的代码不完全是那个提交

最好的那个网络（`mlp_standardised`）再用**全部**训练蛋白训练一次，把模型存进 MLflow。交叉验证里的 5 个模型各自少看了一折，只用来打分；真正拿去用的是看过全部训练数据的这一个。

### 执行
约 4 分钟。每个运行结束时 MLflow 会打印一个链接，点开就是它在 Databricks 上的那一页：

In [ ]:
REFERENCES = {"zero_shot": tracking.read_scores(w, "esm2_zero_shot"), "baseline": tracking.read_scores(w, "baseline")}   # 07f 的两根横杆
SIGNATURE = tracking.tensor_signature({"embedding": 1280, "features": 16}, 20)   # 模型的输入输出长什么样
BEST = "mlp_standardised"
predictions = {}
for name, config in neural.RUNS.items():
    predictions[name] = neural.cross_validate(train, table, index, **config)
    final = neural.fit(train, table, index, **config) if name == BEST else None   # 最好的那个：用全部训练蛋白再训练一次，存起来
    tracking.cv_run(w, EXPERIMENT, name, {**neural.parameters(config), "feature_table_version": VERSION, "esm2_revision": ESM2},
                    baseline.dataset_scores(predictions[name]), references=REFERENCES,
                    model=final, signature=SIGNATURE if final else None, chapter="08a")
    print(name, baseline.summarise(baseline.dataset_scores(predictions[name])), "\n")
print("存回平台的预测:", f"{tracking.save_predictions(w, predictions[BEST], BEST):,}", "行")

### 结果怎么看
三个运行各打印一行成绩，最后一行是 `841,400 行`：最好那个网络对每个训练突变的预测，已经存进了平台的 `ml.cv_predictions`，和基线的预测放在同一张表里。

### 去网页上看
点任意一个 `View run` 链接，或者 Databricks 左边栏 → **Experiments** → `qafi-variant-effect`：
- 现在列表里 07f 的运行和 08a 的运行**混在一起**，可以勾选 `baseline` 和 `mlp_standardised` → **Compare**
- 点进 `mlp_standardised`：参数里有 `device = mps`、`epochs = 4`；标签里有 `trained_on = laptop`；**Logged models** 里有存好的网络

### 公司里什么情况怎么做
- **结果不能只活在一台笔记本上**。旧 QAFI 代码的教训：换了电脑、清了硬盘，结果就没了。训练可以在任何地方，记录必须在中央
- **`-dirty` 的结果不能当正式结果**。正式的那一次，先提交代码再跑
- 同一张卷子、同一批折，是比较两个模型的前提。很多论文里"我们的模型更好"经不起推敲，就是因为两边的数据划分不一样

### 面试怎么说
> "The network is evaluated on exactly the same grouped folds and with the same scoring code as the tree baseline, so per-assay scores are paired. It trains on a laptop GPU but logs to the central MLflow with the code version, data version and a flag for uncommitted changes."

---

## 第 6 步 · 看结果

### 在干什么
把三个网络和 07f 的两根横杆放在一张表里。

### 执行

In [ ]:
runs = tracking.latest_runs(EXPERIMENT)
runs.loc[[*neural.RUNS][::-1] + ["baseline", "esm2_zero_shot"], ["all", "domainome", "proteingym", "mavedb"]].round(3)

### 结果怎么看
每组实验的 Spearman 的中位数（不同机器上可能差 0.005 左右）：

| 运行 | 全部 | domainome | proteingym | mavedb（功能） |
|---|---|---|---|---|
| `mlp_standardised` | **0.627** | **0.652** | **0.634** | 0.378 |
| `mlp_embedding_features` | 0.593 | 0.626 | 0.599 | 0.370 |
| `mlp_embedding` | 0.585 | 0.614 | 0.594 | 0.375 |
| `baseline`（07f 的树模型） | 0.550 | 0.572 | 0.583 | 0.409 |
| `esm2_zero_shot` | 0.455 | 0.461 | 0.468 | 0.398 |

**每样东西值多少**
- **只用向量**（`mlp_embedding`）就到了 0.585，已经超过用 16 个精心挑选的特征的基线（0.550）。那 1,280 个数里确实有我们手工摘要时丢掉的信息
- **再拼上 16 个特征**：只多了 0.008。结构特征带来的新信息不多，说明 ESM 的向量里已经隐含了不少"埋不埋"这类信息
  - 向量和这 16 个特征**确实是两类东西**。向量来自只读过序列的 ESM-2，描述的是"这个位置"；16 个特征里有 6 个来自 AlphaFold 的三维结构，还有几个描述的是"这次换成了什么"。不同来历的数字可以放在一起，就像医生同时看影像和化验单，前提是先把量级统一（所以向量过 LayerNorm，特征各自标准化）
  - 我另外跑过一个对照，表里没有：只用向量、标签也标准化，是 0.616；加上 16 个特征是 0.627。一组一组配对比较，稳定性平均高 0.009，功能平均高 0.020，都有统计显著性。**有用，但是小头**：0.627 里有 0.616 是向量自己挣的
- **标签各自标准化**：多了 0.034，是三步里最值的一步，而它一个参数都没加

和基线一对一比，一类数据一行：

In [ ]:
tracking.comparison(runs, "mlp_standardised", "baseline")

### 结果怎么看
| 数据 | 网络更好的实验占 | 平均高多少 | p 值 | 结论 |
|---|---|---|---|---|
| domainome | 90% | +0.079 | < 0.001 | **网络明显更好** |
| proteingym | 80% | +0.056 | 0.01 | **网络明显更好** |
| mavedb（功能） | 33% | −0.017 | 0.005 | **网络明显更差** |

**又一次，要说实话：**

> 稳定性这门课，网络比基线又高了一大截（0.572 → 0.652）。功能这门课，网络不但没过线，还比基线和零样本都**差了一点**。

为什么更强的模型在功能上反而更差：
- 训练数据里 445 组实验有 382 组测的是稳定性。网络的本事越大，就把"稳定性"这件事学得越彻底，它的预测也就越像一个**稳定性预测器**
- 可是功能不只是稳定。一个残基露在表面、换了也不影响折叠，网络就判它"没事"；但它可能正好是催化位点。零样本的 ESM 分数不懂稳定性，却懂"这个位置进化上动不得"，所以在功能上反而不吃亏
- 我还试过给功能数据单开一个输出头、把零样本分数当起点让网络只学修正量，都没能让功能这一栏超过零样本

也就是说：**这两个模型各懂一半**。下一步就从这里来。

### 公司里什么情况怎么做
- 一个模型在总分上赢了，要立刻问：**在哪一块上输了？** 这里总分从 0.550 涨到 0.627，谁看了都高兴，但功能那一块其实退步了
- 训练数据的构成决定模型学成什么样。数据里八成是稳定性，模型就是个稳定性模型，再强的网络也改变不了这一点

### 面试怎么说
> "A frozen-ESM head beat the tree baseline clearly on stability-type assays, 0.65 versus 0.57, but was slightly worse on functional assays. With 86% of training assays measuring abundance or stability, a stronger learner becomes a better stability predictor, which is not the same as a functional-effect predictor."

---

## 第 7 步 · 合起来用

### 在干什么
把零样本、基线、网络三个预测合成一个，看功能这一栏能不能过线。

### 背景
比喻：**三个大夫会诊**。一个只看进化史（零样本），一个看 16 项体检指标（树模型），一个读完整的 ESM 向量（网络）。三个人各有盲区，但盲区不一样。让他们各自给病人排个轻重顺序，再把三个顺序平均一下，往往比任何一个人单独判断都准。这叫**集成**（ensemble）。

### 用了什么技术，为什么
**按排名平均**（rank average）：
- 三个预测的单位都不一样（一个是对数概率，一个是 0–1 的标签，一个是标准化后的分数），不能直接相加
- 所以在每组实验内部，先把每个预测换成**名次**（0 到 1），再把三个名次取平均
- 没有任何要训练的参数，三个模型各占一票，所以不存在"集成本身过拟合"的问题

基线的预测在平台的 `ml.cv_predictions` 里（07f 存的），取下来和网络的预测按突变对齐。

### 执行

In [ ]:
lakehouse.download(w, f"SELECT * FROM {CATALOG}.ml.cv_predictions WHERE run_name = 'baseline'", MODELS / "baseline_predictions.parquet")
trees = pd.read_parquet(MODELS / "baseline_predictions.parquet").rename(columns={"predicted": "trees"})[[*baseline.KEY, "trees"]]
members = predictions[BEST].rename(columns={"predicted": "mlp"}).merge(trees, on=baseline.KEY)
ensemble = members.assign(predicted=baseline.rank_average(members, ["esm_delta", "trees", "mlp"]))
tracking.cv_run(w, EXPERIMENT, "ensemble", {"model": "rank average", "members": "esm2_zero_shot, baseline, mlp_standardised", "feature_table_version": VERSION},
                baseline.dataset_scores(ensemble), references=REFERENCES, chapter="08a")
tracking.save_predictions(w, ensemble, "ensemble")
runs = tracking.latest_runs(EXPERIMENT)
display(runs.loc[["ensemble", BEST, "baseline", "esm2_zero_shot"], ["all", "domainome", "proteingym", "mavedb"]].round(3))
tracking.comparison(runs, "ensemble", "zero_shot")

### 结果怎么看
| 运行 | 全部 | domainome | proteingym | mavedb（功能） |
|---|---|---|---|---|
| `ensemble` | 0.592 | 0.617 | 0.613 | **0.432** |
| `mlp_standardised` | **0.627** | **0.652** | **0.634** | 0.378 |
| `baseline` | 0.550 | 0.572 | 0.583 | 0.409 |
| `esm2_zero_shot` | 0.455 | 0.461 | 0.468 | 0.398 |

第二张表是集成和零样本的配对比较。功能那一行：

| 数据 | 集成更好的实验占 | 平均高多少 | p 值 |
|---|---|---|---|
| mavedb（功能） | 67% | +0.024 | 0.001 |

- **功能这一栏过线了**：0.432，比零样本的 0.398 高，63 组功能实验里三分之二变好，p = 0.001。07f 定的目标是超过 0.40
- 和基线比，集成在功能上也更好：89% 的功能实验更高，平均 +0.027
- **稳定性上，集成反而不如网络单独用**（0.617 对 0.652）：零样本在稳定性上弱，拉了后腿

所以答案不是一个模型，而是**两个问题各有各的答案**：

| 想问什么 | 用哪个 | 成绩 |
|---|---|---|
| 这个突变会不会让蛋白**不稳定**？ | `mlp_standardised` | 0.65 |
| 这个突变会不会让蛋白**失去功能**？ | `ensemble` | 0.43 |

这在生物学上说得通：蛋白要有功能，先得折叠得住（网络管这个），关键位置还得是对的氨基酸（零样本管这个）。两样都要，所以合起来才行。

**要诚实说明的两点**
- 功能上的提升是真的，但不大：0.398 → 0.432。07b 说的瓶颈没有变：有功能数据的训练蛋白只有 29 个
- 我试过几种组合（两个模型的、三个模型的、不同的权重），方向都一致，最后用了最朴素的"三个各一票"。这些比较都发生在训练蛋白的交叉验证里；真正的验收是 09 号在考试蛋白上的那一次

### 公司里什么情况怎么做
- **集成是提升成绩最便宜的办法**，前提是成员的错误不一样。三个差不多的模型合起来没用；一个懂进化、一个懂结构、一个懂向量，才有用
- 代价是上线要维护三个模型。值不值，看多出来的那点成绩对业务有多重要
- **一个产品可以给多个分数**。只给一个"致病性分数"简单，但把"不稳定"和"失去功能"分开报，对想知道**为什么**的用户更有用

### 面试怎么说
> "No single model won everywhere, so the answer is two outputs: the embedding network for stability, and a parameter-free rank average of zero-shot ESM-2, the tree model and the network for functional effect, which beat zero-shot on two thirds of functional assays with p of 0.001. The members fail differently, which is why averaging helps."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 08a PyTorch network on ESM-2 embeddings and a rank-average ensemble") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 PyTorch | 张量、自动求导、模块、优化器 | 训练 = 前向、损失、反向、改一步 |
| 2 数据 | 训练突变和向量取到本机 | 考试蛋白不下载；记下版本 |
| 3 网络 | 冻结的 ESM-2 后面接一个 73 万参数的头 | 迁移学习；缩放常数存在模型里 |
| 4 训练 | 看两条曲线，轮数定在 4 | 训练损失下降不等于学会了：过拟合 |
| 5 考试 | 三个网络，同一张卷子，记进 MLflow | 本机训练，中央记账 |
| 6 结果 | 稳定性 0.652，功能 0.378 | 总分赢了，要问哪一块输了 |
| 7 集成 | 三个预测按排名平均，功能 0.432 | 两个问题，两个答案 |

**一句话**：把 ESM-2 的整个向量交给一个小网络，稳定性预测大幅提高；功能预测要靠三个模型合起来才过线。

## 面试题

1. **What happens in one training step?**
   > "Forward pass on a batch, compute the loss, backward pass to get every parameter's gradient by automatic differentiation, optimizer step. One pass over the data is an epoch."

2. **How do you detect and handle overfitting?**
   > "Plot training loss against a held-out score per epoch. When the loss keeps falling but the held-out score turns down, stop there; dropout and weight decay help, but for us the number of epochs mattered most."

3. **What is transfer learning and why freeze the backbone?**
   > "Reusing a large pretrained model's representations and training only a small head on top. Freezing is cheap, needs little data and cannot damage the pretrained knowledge; fine-tuning the backbone is the next step only if the head is not enough."

4. **Your new model has a better overall score. Do you ship it?**
   > "Not before looking at segments. Ours improved the average by 0.08 but got worse on functional assays, because most training data measured stability. We ended up serving two outputs."

5. **When does an ensemble help?**
   > "When its members make different errors. Ours combine evolutionary constraint, hand-made structure features and learned embeddings by averaging ranks, with no fitted weights, and that was the only thing that beat zero-shot on functional assays."

**下一步 08b · 图神经网络**：这一章的网络一次只看**一个残基**的向量，看不见它在三维空间里挨着谁。功能位点往往是空间上凑在一起的几个残基。08b 把 07d 存好的"蛋白的图"用上，让每个残基在做判断前先看看邻居。